In [ ]:
library(Seurat)
library(yaml)
library(dplyr)
library(stringr)

yaml_path <- "ovary_markers.yaml"
rds_path <- "roi_1/roi_1_seurat_object_small.RDS"

In [ ]:
seurat <- readRDS(rds_path)
seurat$xenium_clusters <- seurat$clusters
seurat

In [ ]:
names(seurat@meta.data)

In [ ]:
marker_list <- yaml::read_yaml(file = yaml_path)
marker_list <- lapply(marker_list$cell_types, '[[', "markers")
marker_list <- lapply(marker_list, str_to_sentence)
marker_list

In [ ]:
seurat <- AddModuleScore(object = seurat, features = marker_list, assay = "Xenium", slot = "counts", name = "CellTypeScore")

In [ ]:
score_cols <- grep("^CellTypeScore", colnames(seurat@meta.data), value = TRUE)
names(score_cols) <- names(marker_list)

for (i in seq_along(score_cols)){
    colnames(seurat@meta.data)[
        colnames(seurat@meta.data) == score_cols[i]
    ] <- paste0(names(score_cols)[i], "_score")
}

In [ ]:
seurat@meta.data |>
    select(xenium_clusters, ends_with("_score")) |>
    group_by(xenium_clusters) |>
    summarize(across(everything(), mean)) ->
    cluster_scores

In [ ]:
cluster_scores |>
    rowwise() |>
    mutate(
        max_score = max(c_across(ends_with("_score"))),
        predicted_celltype = ifelse(
            max_score > 0.3,
            names(across(ends_with("_score")))[which.max(c_across(ends_with("_score")))],
            "Unassigned"
        )
    ) ->
    annotations

In [ ]:
names(seurat@meta.data)

In [ ]:
VlnPlot(object = seurat, features = "theca_cell_score", group.by = "xenium_clusters")